# FPL Squad Optimizer - Dashboard

This notebook uses the refactored `fpl_engine` modules to run the optimization pipeline. 
It is designed to run in Google Colab or a local Jupyter environment.

In [ ]:
# Run this if you are in Google Colab to mount your drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import sys
    # Replace with your actual path in Drive
    sys.path.append('/content/drive/My Drive/Hobby/FPL')
except ImportError:
    pass  # Not running in Colab

In [ ]:
%load_ext autoreload
%autoreload 2
import asyncio
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

from fpl_engine.data import get_current_players_df, fetch_raw_history_cache, get_team_df, get_pos_constraint_df, get_current_gameweek
from fpl_engine.features import compute_rolling_team_ratings, blend_team_ratings, get_fixture_players_stats_df
from fpl_engine.scoring import _fit_bonus_multinomial, _calculate_performance_indices, create_optimized_custom_score
from fpl_engine.solver import plan_sequential_transfers, create_optimal_fpl_team
from fpl_engine.visualization import _render_sync_plots

## 1. Fetch Data & Build Features

In [ ]:
# 1. Fetch Base Data
current_gw = get_current_gameweek()
fpl_team_df = get_team_df()
players_df = get_current_players_df()

# 2. Fetch Match History (Async)
active_player_ids = players_df[players_df['minutes'] > 0]['id'].tolist()
raw_history_df = asyncio.run(fetch_raw_history_cache(active_player_ids, use_cache=True))

In [ ]:
# 3. Compute Team Ratings & Blends
team_fixture_df, latest_ratings = compute_rolling_team_ratings(raw_history_df)
blended_team_df = blend_team_ratings(team_fixture_df, latest_ratings, fpl_team_df)

# 4. Generate Fixture Specific Projections
fixture_player_df = get_fixture_players_stats_df(players_df, blended_team_df)

## 2. Generate Scoring Projections

In [ ]:
# 1. Fit Bonus Multinomial Model
bonus_model = _fit_bonus_multinomial(raw_history_df)

# 2. Calculate Final Performance Indices (Expected Points)
gw_projection_df = _calculate_performance_indices(fixture_player_df, raw_history_df, bonus_model)

# 3. Add Custom Dynamic Score (Optional)
# gw_projection_df = create_optimized_custom_score(gw_projection_df, current_gw)

## 3. Run Optimization & Visualizations

In [ ]:
# Visualize player metrics and potential
pos_df = get_pos_constraint_df()
_render_sync_plots(gw_projection_df, positions=['GKP', 'DEF', 'MID', 'FWD'])

In [ ]:
# Optimize Team Selection
starter_df, bench_df = create_optimal_fpl_team(budget=100)


In [ ]:
# Optional: Sequential Transfer Planner
# prob, result = plan_sequential_transfers(
#     gw_projection_df,
#     start_gameweek=current_gw + 1,
#     planning_horizon=3,
# )